In [2]:
import re


def remove_stuttering_annotations(text):
    """
    Remove stuttering annotations from transcribed text to get semantic transcription.

    This produces the output that a speech recognition model should generate
    (i.e., what the speaker intended to say without stuttering events).

    Based on the AS-70 paper preprocessing steps (Section 3.2):
    - Removes stuttering event labels (/b, /p, /r, /i)
    - Removes repeated words/characters in square brackets
    - Removes interjection characters marked with /i
    - Removes punctuation
    """

    # First, identify and remove interjection characters marked with /i
    # Pattern: character/i -> remove the character
    text = re.sub(r"([呃啊嗯哦额])\/i", "", text)

    # Handle square brackets with repetitions
    # Pattern: word[word...] means the word in brackets is repeated (should be removed)
    def remove_repetitions(match):
        before_bracket = match.group(1)
        return before_bracket

    text = re.sub(r"([^[\]]+)\[[^\]]+\]", remove_repetitions, text)

    # Remove remaining stuttering markers: /b, /p, /r, /i (and their combinations)
    text = re.sub(r"/[bpri]+", "", text)

    # Remove punctuation
    text = re.sub(r'[，。、！？；：""' "（）《》【】…—]", "", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", "", text)

    return text.strip()


def process_annotation_file(input_filename, output_filename):
    """
    Process an annotation file and create a cleaned version with stuttering removed.
    """
    with open(input_filename, "r", encoding="utf-8") as f:
        lines = f.readlines()

    cleaned_lines = []
    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Split by tab to get timestamp and text
        parts = line.split("\t")
        if len(parts) >= 3:
            start_time = parts[0]
            end_time = parts[1]
            text = parts[2]

            # Clean the text
            cleaned_text = remove_stuttering_annotations(text)

            # Create cleaned line
            cleaned_line = f"{start_time}\t{end_time}\t{cleaned_text}"
            cleaned_lines.append(cleaned_line)

    # Write to output file
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write("\n".join(cleaned_lines))

    return len(cleaned_lines)


# # Process all files
# files_to_process = [
#     ("D0001_A.txt", "D0001_A_cleaned.txt"),
#     ("D0001_B.txt", "D0001_B_cleaned.txt"),
#     ("P0001.txt", "P0001_cleaned.txt"),
# ]

# for input_file, output_file in files_to_process:
#     num_lines = process_annotation_file(input_file, output_file)
#     print(f"✓ Processed {input_file} -> {output_file} ({num_lines} lines)")


In [10]:
import soundfile as sf


def slice_wav(src_path, dst_path, start_sec, end_sec, blocksize=1024):
    with sf.SoundFile(src_path) as infile:
        # Infer parameters
        samplerate = infile.samplerate
        channels = infile.channels
        subtype = infile.subtype

        start_frame = int(start_sec * samplerate)
        end_frame = int(end_sec * samplerate)

        infile.seek(start_frame)

        with sf.SoundFile(
            dst_path, "w", samplerate=samplerate, channels=channels, subtype=subtype
        ) as outfile:
            frames_left = end_frame - start_frame
            while frames_left > 0:
                read_frames = min(blocksize, frames_left)
                data = infile.read(read_frames)
                if len(data) == 0:
                    break
                outfile.write(data)
                frames_left -= len(data)


In [ ]:
from pathlib import Path

raw_audio_dir = Path("/home/benji/dev/ssa/data/as70/raw/Audio")
processed_audio_dir = Path("/home/benji/dev/ssa/data/as70/audio")
if not processed_audio_dir.exists():
    processed_audio_dir.mkdir(parents=True)


processed_rows = []
for path in sorted(Path("/home/benji/dev/ssa/data/as70/raw/Annotation").rglob("*.txt")):
    if not path.is_file():
        continue
    if path.stem.split("_")[1] == "B":  # skip over stuttering event annotations
        continue

    subject_id = path.parent.stem
    print(f"{subject_id=} {path=}")
    lines = path.read_text(encoding="utf-8").splitlines()

    for i, line in enumerate(lines):
        parts = line.split("\t")
        assert len(parts) == 3

        wav_path = raw_audio_dir / f"{subject_id}.wav"
        slice_wav_path = processed_audio_dir / f"{subject_id}_{i:03d}.wav"
        start_sec = float(parts[0])
        end_sec = float(parts[1])
        text = parts[2]

        print(f"Creating wav slice {slice_wav_path} from {start_sec} to {end_sec}")
        slice_wav(wav_path, slice_wav_path, start_sec, end_sec)

        unannotated_text = remove_stuttering_annotations(text)
        is_command = path.stem[0] == "P"

        row = {
            "subject_id": subject_id,
            "annotated_text": text,
            "unannotated_text": unannotated_text,
            "audio_path": str(slice_wav_path),
            "is_command": is_command,
            "start_sec": start_sec,
            "end_sec": end_sec,
        }
        processed_rows.append(row)


subject_id='0001' path=PosixPath('/home/benji/dev/ssa/data/as70/raw/Annotation/0001/D0001_A.txt')
Creating wav slice /home/benji/dev/ssa/data/as70/audio/0001_000.wav from 87.97 to 94.85
